In [ ]:
# Монтирование Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
import shutil
import pandas as pd

In [ ]:
# Клонирование репозитория и подготовка структуры
# Очистка и клонирование
os.chdir('/content')
if os.path.exists('/content/TCT_data'):
    shutil.rmtree('/content/TCT_data')

!git clone https://github.com/zx333445/TCT_data.git
os.chdir('/content/TCT_data')
print(f"В директории: {os.getcwd()}")

# Создание необходимых папок
!mkdir -p /content/TCT_data/data/fold
!mkdir -p /content/TCT_data/models
!mkdir -p /content/TCT_data/logs

# Копирование CSV файлов
!cp /content/drive/MyDrive/НИР/fold/train1.csv /content/TCT_data/data/fold/train.csv
!cp /content/drive/MyDrive/НИР/fold/val1.csv /content/TCT_data/data/fold/val.csv

print("CSV скопированы")

In [ ]:
# Создание вспомогательных модулей
# Создание _utils.py
with open('/content/TCT_data/_utils.py', 'w') as f:
    f.write('''
import torch

def collate_fn(batch):
    return tuple(zip(*batch))
''')

# Создание transforms.py
with open('/content/TCT_data/tool/transforms.py', 'w') as f:
    f.write('''
import torch
import torchvision.transforms as T
from torchvision.transforms import functional as F

class Compose:
    def __init__(self, transforms):
        self.transforms = transforms
    def __call__(self, image, target):
        for t in self.transforms:
            image, target = t(image, target)
        return image, target

class ToTensor:
    def __call__(self, image, target):
        image = F.to_tensor(image)
        return image, target

class ImgAugTransform:
    def __init__(self):
        self.aug = T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(10),
            T.ColorJitter(brightness=0.1, contrast=0.1),
        ])
    def __call__(self, image, target):
        image = self.aug(image)
        return image, target
''')
print("transforms.py создан")


In [ ]:
# Создание класса Dataset
with open('/content/TCT_data/datasets.py', 'w') as f:
    f.write('''
import os
import torch
from PIL import Image
import pandas as pd
from torch.utils.data import Dataset

class TCTDataset(Dataset):
    def __init__(self, root, transforms, train=True, csv_name="train.csv"):
        super().__init__()
        self.root = root
        self.transforms = transforms
        self.train = train

        csv_path = os.path.join(self.root, csv_name)
        print(f"Читаем CSV: {csv_path}")
        annotation = pd.read_csv(csv_path)

        self.base_path = '/content/drive/MyDrive/НИР/data/JPEGImages/'
        self.image_list = []
        self.annotations = []

        for _, row in annotation.iterrows():
            img_name = os.path.basename(row['image_path'])
            new_path = os.path.join(self.base_path, img_name)
            self.image_list.append(new_path)
            self.annotations.append(row['annotation'])

        print(f"   Загружено {len(self.image_list)} изображений")

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_path = self.image_list[idx]

        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Ошибка загрузки {img_path}: {e}")
            image = Image.new('RGB', (1024, 1024), color='black')

        annotation = self.annotations[idx]

        if type(annotation) != str:
            annotation = str(annotation)

        boxes = []
        labels = []

        if self.train and annotation and annotation != 'nan':
            annotation_list = annotation.split(';')

            for anno in annotation_list:
                anno = anno.strip()
                if not anno:
                    continue

                if len(anno) > 2:
                    anno = anno[2:]

                parts = anno.split()

                if len(parts) >= 4:
                    try:
                        x = []
                        y = []
                        for i in range(len(parts)):
                            if i % 2 == 0:
                                x.append(float(parts[i]))
                            else:
                                y.append(float(parts[i]))

                        xmin, xmax = min(x), max(x)
                        ymin, ymax = min(y), max(y)

                        if xmin < xmax and ymin < ymax:
                            boxes.append([xmin, ymin, xmax, ymax])
                            labels.append(1)
                    except:
                        continue

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros(0, dtype=torch.int64)
            area = torch.zeros(0, dtype=torch.float32)
            iscrowd = torch.zeros(0, dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
            iscrowd = torch.zeros(len(boxes), dtype=torch.int64)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx]),
            'area': area,
            'iscrowd': iscrowd
        }

        if self.transforms:
            image, target = self.transforms(image, target)

        return image, target, img_path
''')
print("datasets.py создан")


In [ ]:
# Подготовка файла для валидации mAP
df = pd.read_csv('/content/TCT_data/data/fold/val.csv')
df_map = pd.DataFrame()
df_map['image_path'] = df['image_path'].apply(
    lambda x: f'/content/drive/MyDrive/НИР/data/JPEGImages/{os.path.basename(x)}'
)
df_map['annotation'] = df['annotation']
df_map.to_csv('/content/TCT_data/data/fold/val_for_map.csv', index=False)
print("val_for_map.csv создан")

In [ ]:
# Создание trainer.py
with open('/content/TCT_data/trainer.py', 'w') as f:
    f.write('''
import math
import sys
import os
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
import copy

from tool.voc_eval_new import custom_voc_eval

def freeze_bn(m):
    classname = m.__class__.__name__
    if classname.find("BatchNorm") != -1:
        m.eval()
        m.weight.requires_grad = False
        m.bias.requires_grad = False

def train_one_epoch(epoch, model, loader, optimizer, lr_scheduler, device, writer=None):
    model.train()
    model.apply(freeze_bn)
    train_loss = 0.0

    for i, (images, targets, _) in enumerate(tqdm(loader), 0):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        loss_value = losses.item()
        train_loss += loss_value

        if not math.isfinite(loss_value):
            print("Loss is {}, stopping training".format(loss_value))
            print(loss_dict)
            sys.exit(1)

        if (i+1) % 100 == 0:
            print(f'batch{i+1} loss:{loss_value:.4f}')

        losses.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    avg_loss = train_loss/len(loader)
    print(f'epoch{epoch} loss:{avg_loss:.4f}')
    if writer:
        writer.add_scalar('train/loss', avg_loss, epoch)
    return avg_loss

def validate(epoch, model, loader, device, save_model_path, fold):
    model.eval()
    model.roi_heads.score_thresh = 0.5

    pred_dict = {}
    total_gt = 0
    total_pred = 0

    for images, targets, img_paths in tqdm(loader, desc="Validation"):
        images = [img.to(device) for img in images]
        with torch.no_grad():
            outputs = model(images)

        for output, target, img_path in zip(outputs, targets, img_paths):
            total_gt += len(target['boxes'])

            if len(output['boxes']) > 0:
                boxes = output['boxes'].detach().cpu().numpy()
                scores = output['scores'].detach().cpu().numpy()
                total_pred += len(boxes)

                for box, score in zip(boxes, scores):
                    x1, y1, x2, y2 = box
                    pred_str = f"1 {score:.4f} {x1:.1f} {y1:.1f} {x2:.1f} {y2:.1f}"
                    if img_path not in pred_dict:
                        pred_dict[img_path] = []
                    pred_dict[img_path].append((score, pred_str))

    print(f"\\n GT: {total_gt}, Pred: {total_pred}")

    pred_csv = f'/content/TCT_data/val_pred_fold{fold}.csv'
    with open(pred_csv, 'w') as f:
        f.write('image_path,annotation\\n')
        for img_path, preds in pred_dict.items():
            preds.sort(key=lambda x: x[0], reverse=True)
            pred_strings = [p[1] for p in preds[:100]]
            if pred_strings:
                annotation = ";".join(pred_strings)
                f.write(f'{img_path},{annotation}\\n')
            else:
                f.write(f'{img_path},\\n')

    try:
        gt_csv = '/content/TCT_data/data/fold/val_for_map.csv'
        result = custom_voc_eval(
            gt_csv=gt_csv,
            pred_csv=pred_csv,
            label_list=['1'],
            ovthresh=0.5,
            use_07_metric=False
        )
        mAP = float(result[2]) if len(result) >= 3 else float(result[0])
        print(f" Epoch {epoch} - mAP@0.5: {mAP:.4f} ({mAP*100:.2f}%)")
    except Exception as e:
        print(f" mAP error: {e}")
        mAP = 0.0

    return mAP

def main_process(model, optimizer, lr_sche, dataloaders, num_epochs,
                  use_tensorboard, device, save_model_path, fold, writer=None):

    best_score = 0.0
    best_epoch = 0

    drive_models_dir = '/content/drive/MyDrive/TCT_models_simsiam'
    os.makedirs(drive_models_dir, exist_ok=True)
    print(f" SimSiam модели будут сохраняться в: {drive_models_dir}")

    for epoch in range(num_epochs):
        train_loss = train_one_epoch(epoch, model, dataloaders['train'], optimizer, lr_sche, device, writer)

        if epoch % 5 == 0 or epoch == num_epochs-1:
            val_mAP = validate(epoch, model, dataloaders['val'], device, save_model_path, fold)

            drive_path = f'{drive_models_dir}/simsiam_fold{fold}_epoch{epoch}_map{val_mAP:.4f}.pth'
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'mAP': val_mAP,
                'train_loss': train_loss
            }, drive_path)
            print(f" СОХРАНЕНО НА DRIVE: simsiam_fold{fold}_epoch{epoch}_map{val_mAP:.4f}.pth")

            if val_mAP > best_score:
                best_score = val_mAP
                best_epoch = epoch
                best_stat_dict = copy.deepcopy(model.state_dict())

                torch.save({
                    'epoch': epoch,
                    'model_state_dict': best_stat_dict,
                    'mAP': best_score,
                    'train_loss': train_loss
                }, save_model_path.format(epoch))

                best_drive_path = f'{drive_models_dir}/simsiam_fold{fold}_BEST_map{best_score:.4f}.pth'
                torch.save(best_stat_dict, best_drive_path)
                print(f" BEST SIMSIAM MODEL SAVED TO DRIVE: simsiam_fold{fold}_BEST_map{best_score:.4f}.pth")

            if use_tensorboard:
                writer.add_scalar('Validation/mAP', val_mAP, global_step=epoch)

    print(f"\\n Training Done! Best mAP: {best_score:.4f} ({best_score*100:.2f}%) at epoch {best_epoch}")
    print(f" Все модели сохранены в: {drive_models_dir}")

    if use_tensorboard:
        writer.close()
''')
print("trainer.py создан")


In [ ]:
# Создание train.py
with open('/content/TCT_data/train.py', 'w') as f:
    f.write('''
#!/usr/bin/env python
import torch
import torchvision
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import argparse
import os
import datetime
import time

from tool import transforms as T
import _utils
from datasets import TCTDataset
from trainer import main_process
from network.backbone_utils import resnet_fpn_backbone
from network.faster_rcnn_framework import FasterRCNN
from network.rpn_function import AnchorsGenerator

def create_model(backbone_path=None, device='cpu'):
    print("\\n===============Creating SimSiam model===============")

    backbone = resnet_fpn_backbone('resnet50', pretrained=False)

    if backbone_path and os.path.exists(backbone_path):
        print(f" Загружаем SimSiam веса из: {backbone_path}")

        checkpoint = torch.load(backbone_path, map_location=device)

        if isinstance(checkpoint, dict):
            if 'model_state_dict' in checkpoint:
                state_dict = checkpoint['model_state_dict']
            elif 'state_dict' in checkpoint:
                state_dict = checkpoint['state_dict']
            else:
                state_dict = checkpoint
        else:
            state_dict = checkpoint

        missing, unexpected = backbone.load_state_dict(state_dict, strict=False)
        print(f" SimSiam backbone загружен!")
        print(f"   Missing keys: {len(missing)}")
        print(f"   Unexpected keys: {len(unexpected)}")
    else:
        print(" SimSiam weights not found, using random initialization")

    anchor_sizes = ((32,), (64,), (128,), (256,), (512,))
    aspect_ratios = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
    rpn_anchor_generator = AnchorsGenerator(anchor_sizes, aspect_ratios)

    roi_pooler = torchvision.ops.MultiScaleRoIAlign(
        featmap_names=['0', '1', '2', '3'],
        output_size=7,
        sampling_ratio=2
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=2,
        rpn_anchor_generator=rpn_anchor_generator,
        box_roi_pool=roi_pooler
    )
    print(f" Модель создана")
    return model

parser = argparse.ArgumentParser(description="TCT SIMSIAM")
parser.add_argument("--data_root", default="/content/TCT_data/data/fold", type=str)
parser.add_argument("--fold", default=1, type=int)
parser.add_argument("--train_batch_size", default=32, type=int)
parser.add_argument("--val_batch_size", default=1, type=int)
parser.add_argument("--num_workers", default=2, type=int)
parser.add_argument("--num_epochs", default=30, type=int)
parser.add_argument("--backbone_path", default=None, type=str)
parser.add_argument("--save_model_path", default="/content/TCT_data/models/simsiam_fold1_epoch_{}.pth", type=str)

subparsers = parser.add_subparsers(dest="optimizer_type")
subparsers.required = True

adamw_parser = subparsers.add_parser("AdamW")
adamw_parser.add_argument("--lr", default=8e-5, type=float)
adamw_parser.add_argument("--weight_decay", default=1e-4, type=float)

args = parser.parse_args()

def main(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    print(f"Batch size: {args.train_batch_size}")
    print(f"Epochs: {args.num_epochs}")

    data_transform = {
        "train": T.Compose([T.ToTensor(), T.ImgAugTransform()]),
        "val": T.Compose([T.ToTensor()])
    }

    dataset = TCTDataset(
        root=args.data_root,
        transforms=data_transform["train"],
        train=True,
        csv_name="train.csv"
    )

    dataset_val = TCTDataset(
        root=args.data_root,
        transforms=data_transform["val"],
        train=False,
        csv_name="val.csv"
    )

    dataloader = DataLoader(
        dataset,
        batch_size=args.train_batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        collate_fn=_utils.collate_fn
    )

    dataloader_val = DataLoader(
        dataset_val,
        batch_size=args.val_batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        collate_fn=_utils.collate_fn
    )

    dataloaders = {"train": dataloader, "val": dataloader_val}

    model = create_model(backbone_path=args.backbone_path, device=device)
    model.to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        params,
        lr=args.lr,
        weight_decay=args.weight_decay
    )

    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, args.num_epochs)

    logdir = "./logs_simsiam"
    os.makedirs(logdir, exist_ok=True)
    writer = SummaryWriter(logdir)

    print("\\n===============Start training===============")
    start_time = time.time()

    main_process(
        model=model,
        optimizer=optimizer,
        lr_sche=lr_scheduler,
        dataloaders=dataloaders,
        num_epochs=args.num_epochs,
        use_tensorboard=True,
        device=device,
        save_model_path=args.save_model_path,
        fold=args.fold,
        writer=writer
    )

    total_time = time.time() - start_time
    total_time_str = str(datetime.timedelta(seconds=int(total_time)))
    print(f'\\nTraining time: {total_time_str}')

if __name__ == "__main__":
    main(args)
''')
print("train.py создан")

In [ ]:
# Финальная проверка данных
import sys
sys.path.append('/content/TCT_data')
from tool import transforms as T
from datasets import TCTDataset
import os

print(" ПРОВЕРКА ДАННЫХ")
dataset_train = TCTDataset(
    '/content/TCT_data/data/fold',
    transforms=T.Compose([T.ToTensor()]),
    train=True,
    csv_name="train.csv"
)

total_boxes = 0
print("\n Первые 5 изображений:")
for i in range(min(5, len(dataset_train))):
    img, target, path = dataset_train[i]
    num_boxes = len(target['boxes'])
    total_boxes += num_boxes
    print(f"{i+1}. {os.path.basename(path)} - {num_boxes} боксов")

print(f"\n Всего боксов в первых 5: {total_boxes}")

if total_boxes > 0:
    print("\n Данные загружены корректно. Запуск обучения...")
else:
    print("\n Проблема с загрузкой данных!")

In [ ]:
# ЗАПУСК ОБУЧЕНИЯ SIMSIAM
# Запуск обучения SimSiam с предобученными весами
!cd /content/TCT_data && python train.py \
    --data_root /content/TCT_data/data/fold \
    --train_batch_size 32 \
    --num_epochs 30 \
    --num_workers 2 \
    --backbone_path /content/drive/MyDrive/checkpoints/continue_from_epoch9/faster_rcnn_results/checkpoints/fold1_epoch_5.pth \
    --save_model_path "/content/TCT_data/models/simsiam_fold1_epoch_{}.pth" \
    AdamW \
    --lr 8e-5 \
    --weight_decay 1e-4